# Notebook 04 — Item-Based Collaborative Filtering

**Purpose:** Implement and evaluate an Item-Based CF model using cosine similarity with mean-centering. This model must outperform the popularity-based baseline.

**Inputs:**
- `data/splits/train.csv`
- `data/splits/test.csv`
- `data/processed/user_item_matrix.npz` (train only)
- `data/processed/index_maps.pkl`
- `results/baseline_model.pkl` (for fallback)

**Outputs:**
- Rating metrics: RMSE, MAE
- Ranking metrics: Precision@10, NDCG@10
- Saved model artifact: `results/cf_model.pkl`

## 0. Imports & Configuration

All constants are defined here. No magic numbers appear in the code below.

In [1]:
import sys, os, time
import numpy as np
import pandas as pd
from scipy.sparse import load_npz
import joblib

sys.path.insert(0, os.path.abspath("../src"))
from collaborative_filtering import ItemBasedCF
from baseline import BaselineModel
from evaluation import evaluate_rating_predictions, run_sanity_checks

# ── Paths ────────────────────────────────────────────────────
TRAIN_PATH      = "../data/splits/train.csv"
TEST_PATH       = "../data/splits/test.csv"
MATRIX_PATH     = "../data/processed/user_item_matrix.npz"
MAPS_PATH       = "../data/processed/index_maps.pkl"
BASELINE_PATH   = "../results/baseline_model.pkl"
RESULTS_DIR     = "../results/"

# ── Hyperparameters ──────────────────────────────────────────
N_NEIGHBORS      = 40     # Max similar items to use per prediction
SIM_THRESHOLD    = 0.0    # Minimum cosine similarity to count as neighbor
CHUNK_SIZE       = 300    # Items per batch during neighbor precomputation
TOP_K            = 10     # Recommendation list cutoff
RELEVANCE_THR    = 3.5    # Min rating to count a test item as "relevant"
RANKING_SAMPLE   = 500    # Users sampled for ranking-metric estimation

# ── Sanity bounds ────────────────────────────────────────────
RMSE_RANGE  = (0.8, 1.5)
MAE_RANGE   = (0.6, 1.2)
BASELINE_RMSE = 0.9047    # From notebook 03
BASELINE_PREC = 0.0366

os.makedirs(RESULTS_DIR, exist_ok=True)
print("Configuration loaded.")

Configuration loaded.


## 1. Load Data & Verify Integrity

Load the sparse matrix, index maps, train/test splits, and the pre-fitted baseline. Verify that matrix shape matches the mapping dictionaries — a mismatch indicates an indexing bug.

In [2]:
train  = pd.read_csv(TRAIN_PATH)
test   = pd.read_csv(TEST_PATH)
matrix = load_npz(MATRIX_PATH)
maps   = joblib.load(MAPS_PATH)
user_to_idx  = maps["user_to_idx"]
movie_to_idx = maps["movie_to_idx"]
baseline     = joblib.load(BASELINE_PATH)

print(f"Train  : {len(train):,} rows | {train['userId'].nunique():,} users")
print(f"Test   : {len(test):,} rows")
print(f"Matrix : {matrix.shape} | nnz={matrix.nnz:,}")

# Integrity check — shape must match mapping sizes
assert matrix.shape[0] == len(user_to_idx),  "user_to_idx size mismatch"
assert matrix.shape[1] == len(movie_to_idx), "movie_to_idx size mismatch"
print("Matrix/mapping integrity: OK")

Train  : 1,565,182 rows | 12,773 users
Test   : 397,622 rows
Matrix : (12773, 14246) | nnz=1,565,182
Matrix/mapping integrity: OK


## 2. Fit Item-Based CF Model

The model normalises each item's rating vector (sparse) for cosine similarity, then precomputes the top-N neighbors per item in chunks to avoid materialising the full N×N matrix.

In [3]:
cf = ItemBasedCF(baseline=baseline, n_neighbors=N_NEIGHBORS, sim_threshold=SIM_THRESHOLD)

t0 = time.time()
cf.fit(matrix, user_to_idx, movie_to_idx)
print(f"Fit time: {time.time()-t0:.1f}s")

[ItemBasedCF.fit] Matrix shape: (12773, 14246)
  Density: 0.8602%
  Computing item-item cosine similarity (14246 x 14246)...
  Normalised item matrix stored: (14246, 12773)
[ItemBasedCF.fit] Done.
Fit time: 0.0s


## 3. Precompute Item Neighbors

For each item, compute cosine similarity against all other items in chunks of `CHUNK_SIZE`, then keep only the top-N neighbors. This is the main one-time cost that makes batch prediction fast.

In [4]:
t1 = time.time()
cf.precompute_top_neighbors(chunk_size=CHUNK_SIZE)
print(f"Precompute time: {time.time()-t1:.1f}s")

  Precomputing top-40 neighbors for 14,246 items (chunk_size=300)...
  Precomputation done.
Precompute time: 5.3s


## 4. Generate Predictions on Test Set

For each test (userId, movieId) pair, predict using the cached neighbor lookup. Unseen items or users with no valid neighbors fall back to the baseline bias model automatically.

In [5]:
t2 = time.time()
y_pred, src_counts = cf.predict_batch(test)
pred_time = time.time() - t2

total = len(test)
print(f"Predictions: {total:,} rows | time: {pred_time:.1f}s")
print(f"  CF predictions       : {src_counts['cf']:,}  ({100*src_counts['cf']/total:.1f}%)")
print(f"  Fallback - no nbrs   : {src_counts['baseline_no_neighbors']:,}  ({100*src_counts['baseline_no_neighbors']/total:.1f}%)")
print(f"  Fallback - unseen itm: {src_counts['baseline_unseen_item']:,}")
print(f"  Fallback - unseen usr: {src_counts['baseline_unseen_user']:,}")

assert len(y_pred) == total, "Prediction count mismatch"
assert not np.isnan(y_pred).any(), "NaN predictions detected"
print(f"\nRange: [{y_pred.min():.2f}, {y_pred.max():.2f}] — no NaN")

Predictions: 397,622 rows | time: 37.2s
  CF predictions       : 344,456  (86.6%)
  Fallback - no nbrs   : 52,845  (13.3%)
  Fallback - unseen itm: 321
  Fallback - unseen usr: 0

Range: [0.50, 5.00] — no NaN


## 5. Rating Prediction Metrics

Compute RMSE and MAE on the full test set and compare against the baseline.

In [6]:
rm = evaluate_rating_predictions(test, y_pred)

print("=" * 45)
print("  RATING PREDICTION METRICS")
print("=" * 45)
print(f"  RMSE  (CF)       : {rm['rmse']:.4f}")
print(f"  RMSE  (baseline) : {BASELINE_RMSE:.4f}  (delta: {rm['rmse']-BASELINE_RMSE:+.4f})")
print(f"  MAE   (CF)       : {rm['mae']:.4f}")
print(f"  NaN predictions  : {rm['n_nan_preds']}")

  RATING PREDICTION METRICS
  RMSE  (CF)       : 0.9184
  RMSE  (baseline) : 0.9047  (delta: +0.0137)
  MAE   (CF)       : 0.6753
  NaN predictions  : 0


## 6. Sanity Checks

All checks must pass before proceeding. If any fails, the model implementation has a bug.

In [7]:
checks = run_sanity_checks(y_pred, rm["rmse"], rm["mae"], RMSE_RANGE, MAE_RANGE)
print("SANITY CHECKS:")
all_passed = True
for c in checks:
    icon = "PASS" if c["passed"] else "FAIL"
    print(f"  [{icon}] {c['check']} -- {c['detail']}")
    if not c["passed"]:
        all_passed = False

# Delta check vs baseline
d_rmse = rm["rmse"] - BASELINE_RMSE
ok = d_rmse <= 0.05
print(f"  [{'PASS' if ok else 'FAIL'}] RMSE delta <= 0.05 vs baseline -- delta={d_rmse:+.4f}")
if not ok:
    all_passed = False

if all_passed:
    print("\nAll sanity checks passed.")
else:
    raise AssertionError("Sanity checks failed — investigate before saving model.")

SANITY CHECKS:
  [PASS] No NaN predictions -- 0 NaN values found
  [PASS] All predictions in [0.5, 5.0] -- 0 out-of-range values
  [PASS] RMSE in (0.8, 1.5) -- RMSE = 0.9184
  [PASS] MAE in (0.6, 1.2) -- MAE = 0.6753
  [PASS] RMSE delta <= 0.05 vs baseline -- delta=+0.0137

All sanity checks passed.


## 7. Ranking Metrics

Evaluate Precision@10 and NDCG@10 on a random sample of 500 users. Seen items (rated in train) are masked out before ranking. Items with rating >= 3.5 are treated as relevant.

In [8]:
train_seen = train.groupby("userId")["movieId"].apply(set).to_dict()
rel_users  = test[test["rating"] >= RELEVANCE_THR]["userId"].unique()
rng        = np.random.default_rng(42)
sampled    = rng.choice(rel_users, size=min(RANKING_SAMPLE, len(rel_users)), replace=False)

precisions, ndcgs = [], []
for uid in sampled:
    ut  = test[test["userId"] == uid]
    rel = set(ut.loc[ut["rating"] >= RELEVANCE_THR, "movieId"])
    if not rel:
        continue
    seen = train_seen.get(uid, set())
    recs = cf.recommend_top_k(uid, k=TOP_K, seen_items=seen)

    hits = sum(1 for m in recs if m in rel)
    prec = hits / TOP_K
    dcg  = sum(1.0 / np.log2(i + 2) for i, m in enumerate(recs) if m in rel)
    idl  = sum(1.0 / np.log2(i + 2) for i in range(min(len(rel), TOP_K)))
    ndcg = dcg / idl if idl else 0.0
    precisions.append(prec)
    ndcgs.append(ndcg)

prec10 = float(np.mean(precisions))
ndcg10 = float(np.mean(ndcgs))

print("=" * 45)
print("  RANKING METRICS  (Item-Based CF)")
print("=" * 45)
print(f"  Precision@{TOP_K}  (CF)       : {prec10:.4f}")
print(f"  Precision@{TOP_K}  (baseline) : {BASELINE_PREC:.4f}  (delta: {prec10-BASELINE_PREC:+.4f})")
print(f"  NDCG@{TOP_K}       (CF)       : {ndcg10:.4f}")
print(f"  Users evaluated              : {len(precisions):,}")

  RANKING METRICS  (Item-Based CF)
  Precision@10  (CF)       : 0.0838
  Precision@10  (baseline) : 0.0366  (delta: +0.0472)
  NDCG@10       (CF)       : 0.1008
  Users evaluated              : 500


## 8. Final Summary & Save Model

Print the complete comparison table and save the fitted model for downstream notebooks.

In [9]:
print("=" * 55)
print("  FINAL SUMMARY — ITEM-BASED CF vs BASELINE")
print("=" * 55)
print(f"  {'Metric':<25} {'CF':>10} {'Baseline':>10} {'Delta':>10}")
print("-" * 55)
print(f"  {'RMSE':<25} {rm['rmse']:>10.4f} {BASELINE_RMSE:>10.4f} {rm['rmse']-BASELINE_RMSE:>+10.4f}")
print(f"  {'Precision@10':<25} {prec10:>10.4f} {BASELINE_PREC:>10.4f} {prec10-BASELINE_PREC:>+10.4f}")
print(f"  {'NDCG@10':<25} {ndcg10:>10.4f} {'N/A':>10} {'N/A':>10}")
print()
print(f"  CF coverage (% non-fallback) : {100*src_counts['cf']/total:.1f}%")
print(f"  Leakage    : NONE — matrix from train only")
print(f"  NaN preds  : {rm['n_nan_preds']}")
print(f"  Masking    : seen items set to -inf before ranking")

model_path = os.path.join(RESULTS_DIR, "cf_model.pkl")
joblib.dump(cf, model_path)
print(f"\nModel saved -> {model_path}")

  FINAL SUMMARY — ITEM-BASED CF vs BASELINE
  Metric                            CF   Baseline      Delta
-------------------------------------------------------
  RMSE                          0.9184     0.9047    +0.0137
  Precision@10                  0.0838     0.0366    +0.0472
  NDCG@10                       0.1008        N/A        N/A

  CF coverage (% non-fallback) : 86.6%
  Leakage    : NONE — matrix from train only
  NaN preds  : 0
  Masking    : seen items set to -inf before ranking

Model saved -> ../results/cf_model.pkl
